# 09 — LLM cluster features with local Ollama (Colab, Google Drive) — pristine

Тот же эксперимент, что и у чистых GPT/Claude-версий: фиксированные кластеризаторы, один консистентный блок тюнинга и корректная методика сравнения.

Эта версия:
- монтирует Google Drive;
- устанавливает и запускает локальный Ollama в Colab;
- сохраняет веса Ollama и артефакты на Google Drive;
- переиспользует кэш `llm_only`, если он уже есть;
- сравнивает 5 сценариев:
  - `llm_only`
  - `cluster_before_llm__gmm_diag_5`
  - `cluster_before_llm__mbkm_2`
  - `llm_then_cluster__gmm_diag_5`
  - `llm_then_cluster__mbkm_2`

**Эксперимент (free-features):** список признаков не фиксируется в промпте — LLM сама решает, какие числовые эвристические признаки вернуть. Downstream собирает union колонок и отбрасывает слишком разреженные.


In [ ]:
import os, sys, subprocess, shutil, time
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/llm_local_colab_runs/ollama_local_free_features")
INPUT_DIR = DRIVE_PROJECT_DIR / "inputs"
INPUT_DIR.mkdir(parents=True, exist_ok=True)

OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2:3b")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434")

os.environ["OLLAMA_MODELS"] = str(DRIVE_PROJECT_DIR / "ollama_models")
Path(os.environ["OLLAMA_MODELS"]).mkdir(parents=True, exist_ok=True)

def _run(cmd, shell=False, check=True):
    print("RUN:", cmd if isinstance(cmd, str) else " ".join(cmd))
    subprocess.run(cmd, shell=shell, check=check)

_run([sys.executable, "-m", "pip", "install", "-q",
      "scikit-learn", "xgboost", "hdbscan", "requests"])

_run("apt-get -qq update", shell=True)
_run("apt-get -qq install -y zstd curl", shell=True)

if shutil.which("ollama") is None:
    _run("curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst | tar --zstd -x -C /usr", shell=True)
else:
    print("Ollama already installed")

_run("ollama -v", shell=True)

_run("pkill -f 'ollama serve' || true", shell=True, check=False)
_run("nohup ollama serve > /tmp/ollama.log 2>&1 &", shell=True)
time.sleep(8)
_run(f"curl -s {OLLAMA_BASE_URL}/api/tags || true", shell=True, check=False)

print("Pulling Ollama model:", OLLAMA_MODEL)
_run(["ollama", "pull", OLLAMA_MODEL])

print("Drive project dir:", DRIVE_PROJECT_DIR)
print("Input dir:", INPUT_DIR)
print("Ollama model cache dir:", os.environ["OLLAMA_MODELS"])
print("Ollama API:", OLLAMA_BASE_URL)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import re
import json
import time
import shutil
import hashlib
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import requests

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error
from sklearn.metrics import pairwise_distances, pairwise_distances_argmin_min
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline

from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge, HuberRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from IPython.display import display

seed = 2
np.random.seed(seed)

try:
    import hdbscan
    HDBSCAN_AVAILABLE = True
except Exception:
    HDBSCAN_AVAILABLE = False

DATA_CANDIDATES = [
    INPUT_DIR / "data_finall_without_TTM.csv",
    DRIVE_PROJECT_DIR / "data_finall_without_TTM.csv",
    Path("./data_finall_without_TTM.csv"),
    Path("/content/data_finall_without_TTM.csv"),
    Path("/mnt/data/data_finall_without_TTM.csv"),
]
DATA_PATH = None

ARTIFACT_DIR = DRIVE_PROJECT_DIR / "artifacts_ollama_local_free_features"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
LLM_CACHE_DIR = DRIVE_PROJECT_DIR / "llm_feature_cache_ollama_local_free_features"
LLM_CACHE_DIR.mkdir(parents=True, exist_ok=True)

target_col = "duration_hours"
cap = 960

OLLAMA_MODEL = globals().get("OLLAMA_MODEL", os.getenv("OLLAMA_MODEL", "llama3.2:3b"))
OLLAMA_BASE_URL = globals().get("OLLAMA_BASE_URL", os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434"))
OLLAMA_CHAT_URL = OLLAMA_BASE_URL + "/api/chat"
OLLAMA_TIMEOUT = 180
OLLAMA_MAX_RETRIES = 3
OLLAMA_NUM_PREDICT = 256

FINAL_TUNED_REFERENCE_CANDIDATES = [
    INPUT_DIR / "final_summary.csv",
    DRIVE_PROJECT_DIR / "final_summary.csv",
    Path("./final_summary.csv"),
    Path("/content/final_summary.csv"),
    Path("/mnt/data/final_summary.csv"),
]

LLM_LIMIT_TRAIN_ROWS = None
LLM_LIMIT_TEST_ROWS = None

def persist_copy_to_drive(src_path, target_name=None):
    src_path = Path(src_path)
    dst = INPUT_DIR / (target_name or src_path.name)
    try:
        if src_path.exists():
            if src_path.resolve() != dst.resolve():
                shutil.copy2(src_path, dst)
            return dst
    except Exception as e:
        print(f"[WARN] Could not copy {src_path} to Drive: {e}")
    return dst

def _ollama_alive():
    try:
        r = requests.get(OLLAMA_BASE_URL + "/api/tags", timeout=3)
        return r.ok
    except Exception:
        return False

def _wait_for_ollama(timeout_s=90):
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        if _ollama_alive():
            return True
        time.sleep(1)
    return False

if not _ollama_alive():
    _OLLAMA_PROC = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if not _wait_for_ollama(timeout_s=90):
        raise RuntimeError("Ollama server did not start inside Colab runtime.")
else:
    print("Ollama server already running")

BACKEND_TAG = "ollama_" + re.sub(r"[^a-zA-Z0-9_]+", "_", OLLAMA_MODEL)

print("HDBSCAN available:", HDBSCAN_AVAILABLE)
print("Ollama model:", OLLAMA_MODEL)
print("Artifact dir:", ARTIFACT_DIR.resolve())
print("Will compare against tuned baseline if final_summary.csv is available")


In [ ]:
for candidate in DATA_CANDIDATES:
    if candidate.exists():
        DATA_PATH = candidate
        break
if DATA_PATH is None:
    raise FileNotFoundError(
        "data_finall_without_TTM.csv not found. Upload it to Colab root or set DATA_CANDIDATES manually."
    )

df = pd.read_csv(DATA_PATH)
persist_copy_to_drive(DATA_PATH)

split = int(len(df) * 0.8)
train_full = df.iloc[:split].copy()
test_full  = df.iloc[split:].copy()
test_typical = test_full[test_full[target_col] < cap].copy()

train_filt = train_full[train_full[target_col] < cap].copy()

meta_Xtr0 = train_filt.drop(columns=[target_col], errors="ignore").reset_index(drop=True)
meta_ytr0 = train_filt[target_col].reset_index(drop=True)

# full holdout
meta_Xte0 = test_full.drop(columns=[target_col], errors="ignore").reset_index(drop=True)
meta_yte0 = test_full[target_col].reset_index(drop=True)

# primary holdout on typical tasks only
meta_Xte_typ0 = test_typical.drop(columns=[target_col], errors="ignore").reset_index(drop=True)
meta_yte_typ0 = test_typical[target_col].reset_index(drop=True)

llm_train_raw = train_filt.reset_index(drop=True).copy()
llm_test_raw  = test_full.reset_index(drop=True).copy()
test_typical_mask = (llm_test_raw[target_col] < cap).reset_index(drop=True)

if LLM_LIMIT_TRAIN_ROWS is not None:
    llm_train_raw = llm_train_raw.iloc[:LLM_LIMIT_TRAIN_ROWS].copy()
    meta_Xtr0 = meta_Xtr0.iloc[:LLM_LIMIT_TRAIN_ROWS].copy()
    meta_ytr0 = meta_ytr0.iloc[:LLM_LIMIT_TRAIN_ROWS].copy()

if LLM_LIMIT_TEST_ROWS is not None:
    llm_test_raw = llm_test_raw.iloc[:LLM_LIMIT_TEST_ROWS].copy()
    meta_Xte0 = meta_Xte0.iloc[:LLM_LIMIT_TEST_ROWS].copy()
    meta_yte0 = meta_yte0.iloc[:LLM_LIMIT_TEST_ROWS].copy()
    test_typical_mask = test_typical_mask.iloc[:LLM_LIMIT_TEST_ROWS].copy()
    meta_Xte_typ0 = meta_Xte0.loc[test_typical_mask].reset_index(drop=True)
    meta_yte_typ0 = meta_yte0.loc[test_typical_mask].reset_index(drop=True)

print("train_full   :", train_full.shape)
print("test_full    :", test_full.shape)
print("test_typical :", test_typical.shape)
print("train_filt   :", train_filt.shape)
print("meta_Xtr0    :", meta_Xtr0.shape)
print("meta_Xte0    :", meta_Xte0.shape, "| full holdout")
print("meta_Xte_typ0:", meta_Xte_typ0.shape, "| primary holdout")
print("llm_train_raw:", llm_train_raw.shape)
print("llm_test_raw :", llm_test_raw.shape)
print("Artifacts will be saved to:", ARTIFACT_DIR.resolve())
print("LLM cache dir:", LLM_CACHE_DIR.resolve())


In [ ]:
LLM_EXCLUDE_COLS = {
    "duration_hours",
    "Ключ", "Задача", "Статус", "Резолюция",
    "Создано", "Дата завершения", "created_dt",
    "Бэклог", "Заблокирован", "В работе", "Раскатка",
    "Merged", "Протестировано", "Ревью",
    "Можно тестировать", "Тестируется",
}
CAT_PREFIXES = ["Приоритет_", "Тип_", "Команда_"]

def is_binary_series(s):
    vals = set(pd.Series(s.dropna().unique()).tolist())
    return vals.issubset({0, 1, 0.0, 1.0, False, True})

def infer_llm_feature_groups(df_part):
    safe_cols = [c for c in df_part.columns if c not in LLM_EXCLUDE_COLS]
    binary_cols = [c for c in safe_cols if is_binary_series(df_part[c])]
    numeric_cols = [c for c in safe_cols if c not in binary_cols]
    cat_groups = {}
    used_bin = set()
    for pref in CAT_PREFIXES:
        cols = [c for c in binary_cols if c.startswith(pref)]
        if cols:
            cat_groups[pref] = cols
            used_bin.update(cols)
    tag_cols = [c for c in binary_cols if c not in used_bin]
    return {"numeric_cols": numeric_cols, "cat_groups": cat_groups, "tag_cols": tag_cols}

feature_groups = infer_llm_feature_groups(llm_train_raw)

def _to_jsonable(v):
    if pd.isna(v): return None
    if isinstance(v, (np.integer,)): return int(v)
    if isinstance(v, (np.floating,)): return float(v)
    if isinstance(v, (np.bool_,)): return bool(v)
    if isinstance(v, pd.Timestamp): return v.isoformat()
    return v

def compact_task_payload(row, groups, max_tags=20):
    out = {"numeric_features": {}, "priority": None, "task_type": None, "team": None, "active_tags": []}
    for c in groups["numeric_cols"]:
        out["numeric_features"][c] = _to_jsonable(row[c])
    for pref, cols in groups["cat_groups"].items():
        active = [c[len(pref):] for c in cols if row[c] == 1]
        val = active[0] if active else None
        if pref == "Приоритет_":
            out["priority"] = val
        elif pref == "Тип_":
            out["task_type"] = val
        elif pref == "Команда_":
            out["team"] = val
    out["active_tags"] = [c for c in groups["tag_cols"] if row[c] == 1][:max_tags]
    return out

print("Numeric cols:", len(feature_groups["numeric_cols"]))
print("Tag cols:", len(feature_groups["tag_cols"]))
compact_task_payload(llm_train_raw.iloc[0], feature_groups)

In [ ]:
LLM_FEATURE_SYSTEM_PROMPT = """
Ты аналитик процессов разработки ПО.
По метаданным задачи сам реши, какие эвристические числовые признаки
наиболее полезны для прогноза длительности выполнения задачи,
и верни их. Не придумывай новые факты и не используй скрытые знания о будущем.
Количество и названия признаков выбираешь сам (рекомендуется 4-8).
Названия признаков — короткие snake_case строки на латинице.
Все значения должны быть числами (int или float).
Старайся из запроса в запрос использовать один и тот же набор имён признаков,
чтобы их можно было агрегировать по датасету.
Верни строго JSON вида {"features": {"<name>": <number>, ...}}.
"""

def make_llm_feature_prompt(task_payload, cluster_context=None):
    payload = {"task_metadata": task_payload, "cluster_context": cluster_context}
    return f"""
Верни JSON строго следующего формата:
{{
  "features": {{
    "<имя_признака_1>": <число>,
    "<имя_признака_2>": <число>
  }}
}}

Сам решай, какие именно числовые эвристические признаки извлечь
из метаданных задачи (сложность, риск координации, тестирование,
неопределённость, срочность, ожидаемая длительность и т.п. — или любые
другие, которые посчитаешь полезными). Значения — числа, имена — snake_case.

Данные:
{json.dumps(payload, ensure_ascii=False, indent=2)}
"""

def _clean_json_text(text):
    if not text:
        return ""
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    return m.group(0).strip() if m else text

def _repair_json_text(text):
    """Пытаемся починить типичные косяки LLM-JSON перед json.loads."""
    if not text:
        return text
    # Убираем висячие запятые перед } и ]
    text = re.sub(r",(\s*[}\]])", r"\1", text)
    # Одинарные кавычки в ключах/значениях -> двойные (грубо, но работает для плоских dict'ов)
    text = re.sub(r"(?<=[{,\s])\'([^\'\n:]+)\'(\s*:)", r'"\1"\2', text)
    text = re.sub(r"(:\s*)\'([^\'\n,}]*)\'", r'\1"\2"', text)
    # Неэкранированные NaN/Infinity -> null, чтобы json.loads не ругался
    text = re.sub(r"\bNaN\b", "null", text)
    text = re.sub(r"\b-?Infinity\b", "null", text)
    return text

def _loads_lenient(text):
    try:
        return json.loads(text)
    except Exception:
        return json.loads(_repair_json_text(text))

def _coerce_feature_dict(data):
    feats = {}
    if isinstance(data, dict):
        src = data["features"] if isinstance(data.get("features"), dict) else data
        for k, v in src.items():
            if k == "features":
                continue
            try:
                feats[str(k)] = float(v)
            except (TypeError, ValueError):
                continue
    return feats

def call_ollama_json(system_prompt, user_prompt):
    last_err = None
    for attempt in range(OLLAMA_MAX_RETRIES):
        try:
            prompt = user_prompt if attempt == 0 else user_prompt + "\n\nReturn valid JSON only."
            payload = {
                "model": OLLAMA_MODEL,
                "stream": False,
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ],
                "options": {
                    "temperature": 0,
                    "num_predict": OLLAMA_NUM_PREDICT,
                },
            }
            r = requests.post(OLLAMA_CHAT_URL, json=payload, timeout=OLLAMA_TIMEOUT)
            r.raise_for_status()
            raw_text = r.json().get("message", {}).get("content", "")
            data = _loads_lenient(_clean_json_text(raw_text))
            feats = _coerce_feature_dict(data)
            return {"features": feats, "parse_error": 0}
        except Exception as e:
            last_err = e
            time.sleep(2 ** attempt)
    # Мягкий отказ: не роняем прогон из-за одной кривой задачи.
    print(f"[warn] Ollama generation failed after {OLLAMA_MAX_RETRIES} retries: {last_err}")
    return {"features": {}, "parse_error": 1}

def cached_llm_json(system_prompt, user_prompt):
    key = hashlib.sha256((BACKEND_TAG + system_prompt + user_prompt).encode("utf-8")).hexdigest()
    path = LLM_CACHE_DIR / f"{key}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    result = call_ollama_json(system_prompt, user_prompt)
    # Кэшируем только успешные ответы, чтобы при ретрае ноутбука неудачи можно было перепройти.
    if not result.get("parse_error"):
        path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    return result

print("Backend ready:", BACKEND_TAG)


In [ ]:

def make_clusterer(name, params):
    if name == "MiniBatchKMeans":
        return MiniBatchKMeans(
            n_clusters=params["n_clusters"],
            random_state=seed,
            n_init=20,
            batch_size=1024
        )
    elif name == "GaussianMixture":
        return GaussianMixture(
            n_components=params["n_components"],
            covariance_type=params["covariance_type"],
            random_state=seed
        )
    else:
        raise ValueError(name)

raw_cluster_scaler = StandardScaler()
raw_Xtr_scaled = raw_cluster_scaler.fit_transform(meta_Xtr0)
raw_Xte_scaled = raw_cluster_scaler.transform(meta_Xte0)

raw_cluster_pca = PCA(n_components=min(30, meta_Xtr0.shape[1]), random_state=seed)
raw_Xtr_embed = raw_cluster_pca.fit_transform(raw_Xtr_scaled)
raw_Xte_embed = raw_cluster_pca.transform(raw_Xte_scaled)

def probs_from_dist(all_d):
    inv = 1.0 / (all_d + 1e-6)
    return inv / inv.sum(axis=1, keepdims=True)

def fit_clusterer_and_assign(name, params, Xtr, Xte):
    est = make_clusterer(name, params)

    if name == "MiniBatchKMeans":
        est.fit(Xtr)
        tr_labels = est.predict(Xtr).astype(int)
        te_labels = est.predict(Xte).astype(int)

        if len(np.unique(tr_labels)) < 2:
            return None

        tr_d = pairwise_distances(Xtr, est.cluster_centers_)
        te_d = pairwise_distances(Xte, est.cluster_centers_)
        tr_proba = probs_from_dist(tr_d)
        te_proba = probs_from_dist(te_d)
        return tr_labels, te_labels, tr_proba, te_proba, "native"

    if name == "GaussianMixture":
        est.fit(Xtr)
        tr_proba = est.predict_proba(Xtr)
        te_proba = est.predict_proba(Xte)

        tr_labels = np.argmax(tr_proba, axis=1).astype(int)
        te_labels = np.argmax(te_proba, axis=1).astype(int)

        if len(np.unique(tr_labels)) < 2:
            return None

        return tr_labels, te_labels, tr_proba, te_proba, "native"

    return None

def build_cluster_meta_features(tr_labels, te_labels, tr_proba=None, te_proba=None):
    tr_feat = pd.DataFrame({"cluster_id": tr_labels})
    te_feat = pd.DataFrame({"cluster_id": te_labels})

    tr_sizes = pd.Series(tr_labels).value_counts().to_dict()
    tr_feat["cluster_size_train"] = pd.Series(tr_labels).map(tr_sizes).fillna(0)
    te_feat["cluster_size_train"] = pd.Series(te_labels).map(tr_sizes).fillna(0)

    tr_feat["is_noise"] = (tr_feat["cluster_id"] == -1).astype(int)
    te_feat["is_noise"] = (te_feat["cluster_id"] == -1).astype(int)

    tr_ohe = pd.get_dummies(tr_feat["cluster_id"], prefix="cluster")
    te_ohe = pd.get_dummies(te_feat["cluster_id"], prefix="cluster").reindex(columns=tr_ohe.columns, fill_value=0)

    tr_feat = pd.concat([tr_feat, tr_ohe], axis=1)
    te_feat = pd.concat([te_feat, te_ohe], axis=1)

    if tr_proba is not None and te_proba is not None:
        for k in range(tr_proba.shape[1]):
            tr_feat[f"cluster_proba_{k}"] = tr_proba[:, k]
            te_feat[f"cluster_proba_{k}"] = te_proba[:, k]

    return tr_feat, te_feat

FIXED_CLUSTER_CONFIGS = [
    {
        "tag": "gmm_diag_5",
        "clusterer": "GaussianMixture",
        "params": {"n_components": 5, "covariance_type": "diag"},
    },
    {
        "tag": "mbkm_2",
        "clusterer": "MiniBatchKMeans",
        "params": {"n_clusters": 2},
    },
]

raw_cluster_variants = {}
raw_cluster_summary_rows = []

for cfg in FIXED_CLUSTER_CONFIGS:
    print("Trying raw clusterer:", cfg)
    result = fit_clusterer_and_assign(cfg["clusterer"], cfg["params"], raw_Xtr_embed, raw_Xte_embed)
    if result is None:
        raise RuntimeError(f"Cluster config failed: {cfg}")

    tr_labels, te_labels, tr_proba, te_proba, fit_mode = result
    tr_feat, te_feat = build_cluster_meta_features(tr_labels, te_labels, tr_proba, te_proba)

    raw_cluster_variants[cfg["tag"]] = {
        "cfg": cfg,
        "fit_mode": fit_mode,
        "tr_labels": tr_labels,
        "te_labels": te_labels,
        "tr_proba": tr_proba,
        "te_proba": te_proba,
        "train_feat_raw": tr_feat,
        "test_feat_raw": te_feat,
    }

    raw_cluster_summary_rows.append({
        "tag": cfg["tag"],
        "clusterer": cfg["clusterer"],
        "params": json.dumps(cfg["params"], ensure_ascii=False),
        "fit_mode": fit_mode,
        "n_clusters_train": int(len(np.unique(tr_labels))),
        "train_rows": int(len(tr_labels)),
        "test_rows": int(len(te_labels)),
    })

raw_cluster_summary_df = pd.DataFrame(raw_cluster_summary_rows)
display(raw_cluster_summary_df)


In [ ]:

def _mode_from_onehot_block(df_part, row_mask, prefix):
    cols = [c for c in df_part.columns if c.startswith(prefix)]
    if not cols:
        return None
    sums = df_part.loc[row_mask, cols].sum(axis=0)
    if sums.max() <= 0:
        return None
    best_col = sums.idxmax()
    return best_col[len(prefix):]

def _top_tags_from_binary_cols(df_part, row_mask, max_tags=10):
    cols = [c for c in df_part.columns if c not in LLM_EXCLUDE_COLS]
    tag_cols = []
    for c in cols:
        if c.startswith("Приоритет_") or c.startswith("Тип_") or c.startswith("Команда_"):
            continue
        if is_binary_series(df_part[c]):
            tag_cols.append(c)
    if not tag_cols:
        return []
    sums = df_part.loc[row_mask, tag_cols].sum(axis=0).sort_values(ascending=False)
    sums = sums[sums > 0]
    return list(sums.head(max_tags).index)

def build_train_cluster_contexts(df_train_raw, tr_labels):
    contexts = {}
    uniq = sorted(np.unique(tr_labels))
    numeric_cols = [c for c in feature_groups["numeric_cols"] if c in df_train_raw.columns]
    for cl in uniq:
        mask = (tr_labels == cl)
        cluster_df = df_train_raw.loc[mask]
        if len(cluster_df) == 0:
            continue
        numeric_summary = {}
        for c in numeric_cols[:15]:
            try:
                numeric_summary[c] = float(cluster_df[c].median())
            except Exception:
                pass
        contexts[int(cl)] = {
            "cluster_id": int(cl),
            "cluster_size_train": int(mask.sum()),
            "is_noise": int(cl == -1),
            "dominant_priority": _mode_from_onehot_block(df_train_raw, mask, "Приоритет_"),
            "dominant_type": _mode_from_onehot_block(df_train_raw, mask, "Тип_"),
            "dominant_team": _mode_from_onehot_block(df_train_raw, mask, "Команда_"),
            "top_tags": _top_tags_from_binary_cols(df_train_raw, mask, max_tags=10),
            "numeric_medians": numeric_summary,
        }
    return contexts

for tag, bundle in raw_cluster_variants.items():
    tr_labels = bundle["tr_labels"]
    te_labels = bundle["te_labels"]
    train_cluster_context_map = build_train_cluster_contexts(llm_train_raw, tr_labels)
    cluster_context_train = [train_cluster_context_map[int(cl)] for cl in tr_labels]
    cluster_context_test = [
        train_cluster_context_map.get(
            int(cl),
            {
                "cluster_id": int(cl),
                "cluster_size_train": 0,
                "is_noise": int(cl == -1),
                "dominant_priority": None,
                "dominant_type": None,
                "dominant_team": None,
                "top_tags": [],
                "numeric_medians": {},
            },
        )
        for cl in te_labels
    ]

    raw_cluster_variants[tag]["cluster_context_train"] = cluster_context_train
    raw_cluster_variants[tag]["cluster_context_test"] = cluster_context_test

print("Prepared train/test cluster contexts for:", list(raw_cluster_variants.keys()))


In [ ]:
def find_existing_feature_file(filename):
    save_path = ARTIFACT_DIR / filename
    candidates = [
        save_path,
        INPUT_DIR / filename,
        DRIVE_PROJECT_DIR / filename,
        Path("./") / filename,
        Path("/content") / filename,
        Path("/mnt/data") / filename,
    ]
    for p in candidates:
        if p.exists():
            return p
    return save_path

def generate_llm_features_for_df(df_part, groups, split_name, variant_name, cluster_contexts=None, save_every=20):
    filename = f"llm_features_{BACKEND_TAG}_{variant_name}_{split_name}.csv"
    resume_path = find_existing_feature_file(filename)
    save_path = ARTIFACT_DIR / filename

    rows = []
    done_ids = set()
    if resume_path.exists():
        existing = pd.read_csv(resume_path)
        existing["parse_error"] = existing.get("parse_error", 1).fillna(1).astype(int)

        ok_existing = existing[existing["parse_error"] == 0].copy()
        retry_existing = existing[existing["parse_error"] != 0].copy()

        done_ids = set(ok_existing["row_id"].astype(int).tolist())
        rows = ok_existing.to_dict("records")

        print(
            f"[RESUME] loaded {len(done_ids)} successful rows from {resume_path}; "
            f"will retry {len(retry_existing)} parse-error rows; saving to {save_path}"
        )


    total = len(df_part)

    for n, (i, r) in enumerate(df_part.iterrows(), start=1):
        if i in done_ids:
            continue
        if n % 10 == 0 or n == 1:
            print(f"[{variant_name}:{split_name}] row {n}/{total}", flush=True)
        task_payload = compact_task_payload(r, groups, max_tags=20)
        cluster_ctx = None if cluster_contexts is None else cluster_contexts[i]
        result = cached_llm_json(
            LLM_FEATURE_SYSTEM_PROMPT,
            make_llm_feature_prompt(task_payload=task_payload, cluster_context=cluster_ctx)
        )
        feats = result.get("features", {}) or {}
        row_out = {"row_id": i, "parse_error": int(result.get("parse_error", 0))}
        for k, v in feats.items():
            row_out[f"llm_{k}"] = v
        rows.append(row_out)
        if len(rows) % save_every == 0:
            pd.DataFrame(rows).to_csv(save_path, index=False)

    out = pd.DataFrame(rows).sort_values("row_id").reset_index(drop=True)
    out.to_csv(save_path, index=False)
    out = out.set_index("row_id")
    numeric_cols = [c for c in out.columns if c != "parse_error"]
    for c in numeric_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    # Признаки, встречающиеся меньше чем у 20% строк, отбрасываем как шум.
    min_fill = max(1, int(0.2 * len(out)))
    keep = [c for c in numeric_cols if out[c].notna().sum() >= min_fill]
    drop = [c for c in numeric_cols if c not in keep]
    if drop:
        print(f"[llm-free] dropping sparse llm cols ({len(drop)}): {drop[:10]}{'...' if len(drop)>10 else ''}")
    out = out[keep + ["parse_error"]]
    medians = out[keep].median(numeric_only=True)
    out[keep] = out[keep].fillna(medians)
    out["parse_error"] = out["parse_error"].fillna(0).astype(int)
    return out


In [ ]:
llm_train_feat_only = generate_llm_features_for_df(llm_train_raw, feature_groups, "train", "llm_only", None)
llm_test_feat_only = generate_llm_features_for_df(llm_test_raw, feature_groups, "test", "llm_only", None)

Xtr_llm_only = pd.concat([meta_Xtr0.reset_index(drop=True), llm_train_feat_only.reset_index(drop=True)], axis=1)
Xte_llm_only_full = pd.concat([meta_Xte0.reset_index(drop=True), llm_test_feat_only.reset_index(drop=True)], axis=1)
Xte_llm_only_typ = Xte_llm_only_full.loc[test_typical_mask].reset_index(drop=True)

print("LLM only train/full/typ:", Xtr_llm_only.shape, Xte_llm_only_full.shape, Xte_llm_only_typ.shape)
print("Train parse_error:", llm_train_feat_only["parse_error"].sum())
print("Test parse_error:", llm_test_feat_only["parse_error"].sum())

In [ ]:

cluster_before_llm_variants = {}

for tag, bundle in raw_cluster_variants.items():
    variant_name = f"cluster_before_llm__{tag}"
    print("\n" + "=" * 95)
    print("Generating features for", variant_name)
    print("=" * 95)

    llm_train_feat_cluster_before = generate_llm_features_for_df(
        llm_train_raw, feature_groups, "train", variant_name, bundle["cluster_context_train"]
    )
    llm_test_feat_cluster_before = generate_llm_features_for_df(
        llm_test_raw, feature_groups, "test", variant_name, bundle["cluster_context_test"]
    )

    Xtr_cluster_before_llm = pd.concat(
        [
            meta_Xtr0.reset_index(drop=True),
            bundle["train_feat_raw"].reset_index(drop=True),
            llm_train_feat_cluster_before.reset_index(drop=True),
        ],
        axis=1,
    )
    Xte_cluster_before_llm_full = pd.concat(
        [
            meta_Xte0.reset_index(drop=True),
            bundle["test_feat_raw"].reset_index(drop=True),
            llm_test_feat_cluster_before.reset_index(drop=True),
        ],
        axis=1,
    )
    Xte_cluster_before_llm_typ = Xte_cluster_before_llm_full.loc[test_typical_mask].reset_index(drop=True)

    cluster_before_llm_variants[tag] = {
        "Xtr": Xtr_cluster_before_llm,
        "Xte_full": Xte_cluster_before_llm_full,
        "Xte_typ": Xte_cluster_before_llm_typ,
        "llm_train_feat": llm_train_feat_cluster_before,
        "llm_test_feat": llm_test_feat_cluster_before,
        "cfg": bundle["cfg"],
    }

    print("Variant:", variant_name)
    print("train/full/typ:", Xtr_cluster_before_llm.shape, Xte_cluster_before_llm_full.shape, Xte_cluster_before_llm_typ.shape)
    print("Train parse_error:", llm_train_feat_cluster_before["parse_error"].sum())
    print("Test parse_error:", llm_test_feat_cluster_before["parse_error"].sum())


In [ ]:
# Fix for free-features: train/test can have different LLM columns.
# This cell replaces the crashed "Кластеризация на пространстве после добавления LLM-фич" cell.

def align_test_to_train_columns(Xtr, Xte, label=""):
    train_cols = list(Xtr.columns)
    missing = [c for c in train_cols if c not in Xte.columns]
    extra = [c for c in Xte.columns if c not in train_cols]

    if missing or extra:
        print(
            f"[ALIGN {label}] adding missing={len(missing)}, dropping extra={len(extra)}; "
            f"missing sample={missing[:5]}, extra sample={extra[:5]}"
        )

    return Xte.reindex(columns=train_cols, fill_value=0)

# Align llm_only test matrix to train matrix before StandardScaler.transform
Xte_llm_only_full = align_test_to_train_columns(
    Xtr_llm_only,
    Xte_llm_only_full,
    "llm_only_full"
)
Xte_llm_only_typ = Xte_llm_only_full.loc[test_typical_mask].reset_index(drop=True)

# Also align already prepared cluster_before variants, so later tuning will not fail
for tag, bundle in cluster_before_llm_variants.items():
    bundle["Xte_full"] = align_test_to_train_columns(
        bundle["Xtr"],
        bundle["Xte_full"],
        f"cluster_before_llm__{tag}"
    )
    bundle["Xte_typ"] = bundle["Xte_full"].loc[test_typical_mask].reset_index(drop=True)

# Кластеризация на пространстве после добавления LLM-фич.
# Здесь используется meta + llm space.
Xtr_llm_space = Xtr_llm_only.copy()
Xte_llm_space = align_test_to_train_columns(
    Xtr_llm_space,
    Xte_llm_only_full.copy(),
    "cluster_after_llm_space"
)

cluster_after_scaler = StandardScaler()
Xtr_llm_scaled = cluster_after_scaler.fit_transform(Xtr_llm_space)
Xte_llm_scaled = cluster_after_scaler.transform(Xte_llm_space)

cluster_after_pca = PCA(n_components=min(30, Xtr_llm_space.shape[1]), random_state=seed)
Xtr_llm_embed = cluster_after_pca.fit_transform(Xtr_llm_scaled)
Xte_llm_embed = cluster_after_pca.transform(Xte_llm_scaled)

llm_then_cluster_variants = {}
cluster_after_summary_rows = []

for cfg in FIXED_CLUSTER_CONFIGS:
    print("\n" + "=" * 95)
    print("Trying cluster-after-LLM:", cfg)
    print("=" * 95)

    result = fit_clusterer_and_assign(
        cfg["clusterer"],
        cfg["params"],
        Xtr_llm_embed,
        Xte_llm_embed
    )
    if result is None:
        raise RuntimeError(f"Cluster-after-LLM config failed: {cfg}")

    aft_tr_labels, aft_te_labels, aft_tr_proba, aft_te_proba, aft_fit_mode = result

    cluster_train_feat_after_llm, cluster_test_feat_after_llm = build_cluster_meta_features(
        aft_tr_labels,
        aft_te_labels,
        aft_tr_proba,
        aft_te_proba
    )

    Xtr_llm_then_cluster = pd.concat(
        [
            meta_Xtr0.reset_index(drop=True),
            llm_train_feat_only.reset_index(drop=True),
            cluster_train_feat_after_llm.reset_index(drop=True),
        ],
        axis=1
    )

    Xte_llm_then_cluster_full = pd.concat(
        [
            meta_Xte0.reset_index(drop=True),
            llm_test_feat_only.reset_index(drop=True),
            cluster_test_feat_after_llm.reset_index(drop=True),
        ],
        axis=1
    )

    Xte_llm_then_cluster_full = align_test_to_train_columns(
        Xtr_llm_then_cluster,
        Xte_llm_then_cluster_full,
        f"llm_then_cluster__{cfg['tag']}"
    )

    Xte_llm_then_cluster_typ = Xte_llm_then_cluster_full.loc[test_typical_mask].reset_index(drop=True)

    tag = cfg["tag"]
    llm_then_cluster_variants[tag] = {
        "Xtr": Xtr_llm_then_cluster,
        "Xte_full": Xte_llm_then_cluster_full,
        "Xte_typ": Xte_llm_then_cluster_typ,
        "cfg": cfg,
        "fit_mode": aft_fit_mode,
    }

    cluster_after_summary_rows.append({
        "tag": tag,
        "clusterer": cfg["clusterer"],
        "params": json.dumps(cfg["params"], ensure_ascii=False),
        "fit_mode": aft_fit_mode,
        "n_clusters_train": int(len(np.unique(aft_tr_labels))),
    })

    print("Variant:", f"llm_then_cluster__{tag}")
    print("train/full/typ:", Xtr_llm_then_cluster.shape, Xte_llm_then_cluster_full.shape, Xte_llm_then_cluster_typ.shape)

cluster_after_summary_df = pd.DataFrame(cluster_after_summary_rows)
display(cluster_after_summary_df)


## Методика сравнения вариантов

- `cv_MAE_internal` — внутренний score `GridSearchCV` на filtered-train.
- Подбор гиперпараметров моделей идёт **только по внутреннему CV**.
- Внешний holdout используется **только для финальной отчётной оценки** после завершения тюнинга.
- `holdout_typical_MAE` — основная внешняя метрика на задачах с `duration_hours < 960`.
- `holdout_full_MAE` — диагностическая stress-метрика на полном holdout.
- Сравниваются 5 сценариев:
  - `llm_only`
  - `cluster_before_llm__gmm_diag_5`
  - `cluster_before_llm__mbkm_2`
  - `llm_then_cluster__gmm_diag_5`
  - `llm_then_cluster__mbkm_2`


In [ ]:
BASELINE_BUILDERS = {
    "Ridge": lambda: Pipeline([("sc", StandardScaler()), ("m", Ridge(random_state=seed))]),
    "Lasso": lambda: Pipeline([("sc", StandardScaler()), ("m", Lasso(random_state=seed, max_iter=20000))]),
    "ElasticNet": lambda: Pipeline([("sc", StandardScaler()), ("m", ElasticNet(random_state=seed, max_iter=20000))]),
    "HuberReg": lambda: Pipeline([("sc", StandardScaler()), ("m", HuberRegressor(max_iter=4000))]),
    "BayRidge": lambda: Pipeline([("sc", StandardScaler()), ("m", BayesianRidge())]),
    "RF": lambda: Pipeline([("m", RandomForestRegressor(random_state=seed, n_jobs=-1))]),
    "ExtraTrees": lambda: Pipeline([("m", ExtraTreesRegressor(random_state=seed, n_jobs=-1))]),
    "GBoost": lambda: Pipeline([("m", GradientBoostingRegressor(random_state=seed))]),
    "XGB": lambda: Pipeline([("m", XGBRegressor(random_state=seed, n_jobs=-1, verbosity=0))]),
    "KNN": lambda: Pipeline([("sc", StandardScaler()), ("m", KNeighborsRegressor())]),
}

untuned_baseline_metrics = {}
for model_name, build_fn in BASELINE_BUILDERS.items():
    est = build_fn()
    est.fit(meta_Xtr0, meta_ytr0)
    pred_typ = est.predict(meta_Xte_typ0)
    pred_full = est.predict(meta_Xte0)
    untuned_baseline_metrics[model_name] = {
        "holdout_typical_MAE": float(mean_absolute_error(meta_yte_typ0, pred_typ)),
        "holdout_full_MAE": float(mean_absolute_error(meta_yte0, pred_full)),
    }

FINAL_TUNED_MAP = {
    "Ridge": "Ridge",
    "Lasso": "Lasso",
    "ElasticNet": "Elastic",
    "HuberReg": "Hub-Reg",
    "BayRidge": "BayRidge",
    "GBoost": "Gboost-Reg",
    "XGB": "XGB_reg",
}

final_summary_path = '/content/drive/MyDrive/llm_local_colab_runs/claude_api/inputs/final_summary.csv'
for candidate in FINAL_TUNED_REFERENCE_CANDIDATES:
    if candidate.exists():
        final_summary_path = candidate
        break

final_tuned_reference = None
if final_summary_path is not None:
    final_tuned_reference = pd.read_csv(final_summary_path)
    print("Using tuned baseline reference from:", final_summary_path)
else:
    print("No final_summary.csv found. Common models will fall back to untuned baseline.")

reference_metrics = {}
for model_name in BASELINE_BUILDERS.keys():
    ext_name = FINAL_TUNED_MAP.get(model_name)
    if final_tuned_reference is not None and ext_name in set(final_tuned_reference["model"]):
        row = final_tuned_reference[final_tuned_reference["model"] == ext_name].iloc[0]
        reference_metrics[model_name] = {
            "source": "tuned_final_summary",
            "holdout_typical_MAE": float(row["holdout_typical_MAE"]),
            "holdout_full_MAE": float(row["holdout_full_MAE"]),
            "cv_MAE": float(row["cv_MAE"]),
        }
    else:
        reference_metrics[model_name] = {
            "source": "untuned_baseline",
            "holdout_typical_MAE": untuned_baseline_metrics[model_name]["holdout_typical_MAE"],
            "holdout_full_MAE": untuned_baseline_metrics[model_name]["holdout_full_MAE"],
            "cv_MAE": np.nan,
        }

def extract_core_estimator(estimator):
    return estimator.steps[-1][1] if isinstance(estimator, Pipeline) else estimator

def extract_model_params(estimator):
    core = extract_core_estimator(estimator)
    clean = {}
    for k, v in core.get_params(deep=False).items():
        clean[k] = v.item() if hasattr(v, "item") else v
    return clean

MODEL_TUNING = {
    "Ridge": (
        Pipeline([("sc", StandardScaler()), ("m", Ridge(random_state=seed))]),
        {"m__alpha": np.logspace(-2, 3, 20).tolist()}
    ),
    "Lasso": (
        Pipeline([("sc", StandardScaler()), ("m", Lasso(random_state=seed, max_iter=100000))]),
        {
            "m__alpha": sorted(set(np.round(np.concatenate([np.logspace(-3, 1, 16), [0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0]]), 6).tolist())),
            "m__selection": ["cyclic", "random"],
        }
    ),
    "ElasticNet": (
        Pipeline([("sc", StandardScaler()), ("m", ElasticNet(random_state=seed, max_iter=100000))]),
        {
            "m__alpha": sorted(set(np.round(np.concatenate([np.logspace(-3, 1, 14), [0.3, 0.5, 0.7, 1.0, 1.5, 2.0]]), 6).tolist())),
            "m__l1_ratio": [0.70, 0.85, 0.90, 0.95, 0.98, 0.995],
        }
    ),
    "HuberReg": (
        Pipeline([("sc", StandardScaler()), ("m", HuberRegressor(max_iter=10000))]),
        {
            "m__epsilon": [1.1, 1.2, 1.35, 1.5, 1.75, 2.0, 2.5],
            "m__alpha": [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0],
        }
    ),
    "BayRidge": (
        Pipeline([("sc", StandardScaler()), ("m", BayesianRidge())]),
        {
            "m__alpha_1": [1e-8, 1e-6, 1e-4, 1e-2],
            "m__alpha_2": [1e-8, 1e-6, 1e-4, 1e-2],
            "m__lambda_1": [1e-8, 1e-6, 1e-4, 1e-2],
            "m__lambda_2": [1e-8, 1e-6, 1e-4, 1e-2],
        }
    ),
    "RF": (
        Pipeline([("m", RandomForestRegressor(random_state=seed, n_jobs=-1))]),
        {
            "m__n_estimators": [300],
            "m__max_depth": [10, 20],
            "m__min_samples_leaf": [1, 5],
            "m__max_features": ["sqrt", 0.5],
        }
    ),
    "ExtraTrees": (
        Pipeline([("m", ExtraTreesRegressor(random_state=seed, n_jobs=-1))]),
        {
            "m__n_estimators": [300],
            "m__max_depth": [10, 20],
            "m__min_samples_leaf": [1, 5],
            "m__max_features": ["sqrt", 0.5],
        }
    ),
    "GBoost": (
        Pipeline([("m", GradientBoostingRegressor(random_state=seed))]),
        [
            {
                "m__loss": ["squared_error", "absolute_error"],
                "m__n_estimators": [100, 150],
                "m__learning_rate": [0.05, 0.10],
                "m__max_depth": [2, 3, 4],
                "m__min_samples_leaf": [1, 4],
                "m__min_samples_split": [2, 8],
                "m__subsample": [0.70, 1.0],
                "m__max_features": [None, 1.0],
            },
            {
                "m__loss": ["huber"],
                "m__alpha": [0.90],
                "m__n_estimators": [100, 150],
                "m__learning_rate": [0.05, 0.10],
                "m__max_depth": [2, 3, 4],
                "m__min_samples_leaf": [1, 4],
                "m__min_samples_split": [2, 8],
                "m__subsample": [0.70, 1.0],
                "m__max_features": [None, 1.0],
            },
        ]
    ),
    "XGB": (
        Pipeline([("m", XGBRegressor(
            objective="reg:absoluteerror",
            eval_metric="mae",
            random_state=seed,
            tree_method="hist",
            n_jobs=-1,
            verbosity=0
        ))]),
        {
            "m__n_estimators": [300, 600],
            "m__learning_rate": [0.02, 0.05],
            "m__max_depth": [2, 3],
            "m__min_child_weight": [5, 10],
            "m__subsample": [0.8, 1.0],
            "m__colsample_bytree": [0.8, 1.0],
            "m__reg_alpha": [0.0, 0.1],
            "m__reg_lambda": [1.0, 5.0],
            "m__gamma": [0.0, 1.0],
        }
    ),
    "KNN": (
        Pipeline([("sc", StandardScaler()), ("m", KNeighborsRegressor())]),
        {
            "m__n_neighbors": [5, 20, 50],
            "m__weights": ["uniform", "distance"],
            "m__p": [1, 2],
        }
    ),
}


MODELS_TO_TUNE = list(MODEL_TUNING.keys())
meta_tune_cv = TimeSeriesSplit(n_splits=3)

display(
    pd.DataFrame([
        {"model": m, **reference_metrics[m]}
        for m in MODELS_TO_TUNE
    ]).sort_values(["holdout_typical_MAE", "holdout_full_MAE"]).reset_index(drop=True)
)

In [ ]:
def run_variant_tuning(Xtr_aug, Xte_typ_aug, Xte_full_aug, title="experiment"):
    rows = []
    print("=" * 95)
    print(title)
    print("=" * 95)

    for model_name in MODELS_TO_TUNE:
        pipe, param_grid = MODEL_TUNING[model_name]
        t0 = time.time()
        grid = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            scoring="neg_mean_absolute_error",
            cv=meta_tune_cv,
            n_jobs=-1,
            refit=True,
            verbose=0,
        )
        grid.fit(Xtr_aug, meta_ytr0)

        pred_typ = grid.best_estimator_.predict(Xte_typ_aug)
        pred_full = grid.best_estimator_.predict(Xte_full_aug)

        holdout_typical_mae = mean_absolute_error(meta_yte_typ0, pred_typ)
        holdout_full_mae = mean_absolute_error(meta_yte0, pred_full)
        rmse_typ = np.sqrt(mean_squared_error(meta_yte_typ0, pred_typ))
        rmse_full = np.sqrt(mean_squared_error(meta_yte0, pred_full))
        mdae_typ = median_absolute_error(meta_yte_typ0, pred_typ)
        mdae_full = median_absolute_error(meta_yte0, pred_full)

        ref = reference_metrics[model_name]
        delta_typ = ref["holdout_typical_MAE"] - holdout_typical_mae
        delta_full = ref["holdout_full_MAE"] - holdout_full_mae

        rows.append({
            "model": model_name,
            "reference_source": ref["source"],
            "reference_typical_MAE": round(ref["holdout_typical_MAE"], 2),
            "reference_full_MAE": round(ref["holdout_full_MAE"], 2),
            "cv_MAE_internal": round(-grid.best_score_, 2),
            "holdout_typical_MAE": round(holdout_typical_mae, 2),
            "holdout_full_MAE": round(holdout_full_mae, 2),
            "delta_typical": round(delta_typ, 2),
            "delta_full": round(delta_full, 2),
            "RMSE_typical": round(rmse_typ, 2),
            "RMSE_full": round(rmse_full, 2),
            "MdAE_typical": round(mdae_typ, 2),
            "MdAE_full": round(mdae_full, 2),
            "best_params": json.dumps(extract_model_params(grid.best_estimator_), ensure_ascii=False),
            "time_s": round(time.time() - t0, 1),
        })
        print(
            f"{model_name:<12s} | cv={(-grid.best_score_):.2f} | "
            f"ref_typ={ref['holdout_typical_MAE']:.2f} | "
            f"typ={holdout_typical_mae:.2f} | full={holdout_full_mae:.2f} | "
            f"Δtyp={delta_typ:+.2f} | src={ref['source']}"
        )

    out = pd.DataFrame(rows).sort_values(
        ["cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"]
    ).reset_index(drop=True)
    display(out)
    return out


In [ ]:
def add_experiment_name(df_res, name):
    tmp = df_res.copy()
    tmp["experiment"] = name
    return tmp

all_result_frames = []

# 1) llm_only
llm_only_tuned_df = run_variant_tuning(
    Xtr_llm_only,
    Xte_llm_only_typ,
    Xte_llm_only_full,
    title="VARIANT 1: llm_only"
)
llm_only_tuned_df.to_csv(ARTIFACT_DIR / "llm_only_tuned_results.csv", index=False)
all_result_frames.append(add_experiment_name(llm_only_tuned_df, "llm_only"))

# 2) cluster_before_llm variants
for tag, bundle in cluster_before_llm_variants.items():
    exp_name = f"cluster_before_llm__{tag}"
    res = run_variant_tuning(
        bundle["Xtr"],
        bundle["Xte_typ"],
        bundle["Xte_full"],
        title=f"VARIANT 2: {exp_name}"
    )
    res.to_csv(ARTIFACT_DIR / f"{exp_name}_tuned_results.csv", index=False)
    all_result_frames.append(add_experiment_name(res, exp_name))

# 3) llm_then_cluster variants
for tag, bundle in llm_then_cluster_variants.items():
    exp_name = f"llm_then_cluster__{tag}"
    res = run_variant_tuning(
        bundle["Xtr"],
        bundle["Xte_typ"],
        bundle["Xte_full"],
        title=f"VARIANT 3: {exp_name}"
    )
    res.to_csv(ARTIFACT_DIR / f"{exp_name}_tuned_results.csv", index=False)
    all_result_frames.append(add_experiment_name(res, exp_name))

llm_compare_df = pd.concat(all_result_frames, ignore_index=True)

# CV-first summary (methodologically primary)
best_by_exp_cv = (
    llm_compare_df
    .sort_values(["cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"])
    .groupby("experiment", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

# Holdout report (descriptive only, no further selection after this)
best_by_exp_holdout = (
    llm_compare_df
    .sort_values(["holdout_typical_MAE", "holdout_full_MAE", "cv_MAE_internal"])
    .groupby("experiment", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

global_holdout_report = llm_compare_df.sort_values(
    ["holdout_typical_MAE", "holdout_full_MAE", "cv_MAE_internal"]
).reset_index(drop=True)

print("Best model in each experiment by internal CV (primary selection table):")
display(best_by_exp_cv)

print("Best model in each experiment by holdout (descriptive report only):")
display(best_by_exp_holdout)

print("Global top-20 by holdout_typical_MAE (report only):")
display(global_holdout_report.head(20))

llm_compare_df.to_csv(ARTIFACT_DIR / "llm_cluster_compare_results.csv", index=False)
best_by_exp_cv.to_csv(ARTIFACT_DIR / "llm_cluster_best_by_experiment_cv.csv", index=False)
best_by_exp_holdout.to_csv(ARTIFACT_DIR / "llm_cluster_best_by_experiment_holdout.csv", index=False)
global_holdout_report.to_csv(ARTIFACT_DIR / "llm_cluster_global_holdout_report.csv", index=False)
raw_cluster_summary_df.to_csv(ARTIFACT_DIR / "raw_cluster_config_summary.csv", index=False)
cluster_after_summary_df.to_csv(ARTIFACT_DIR / "cluster_after_llm_config_summary.csv", index=False)

summary_payload = {
    "selection_rule": "GridSearchCV selects hyperparameters by internal CV only",
    "final_report_rule": "holdout tables are descriptive only after tuning is completed",
    "stress_test_rule": "holdout_full_MAE is diagnostic only",
    "reference_for_common_models": "final_summary.csv tuned baseline when available",
    "fixed_cluster_configs": [
        {"tag": "gmm_diag_5", "clusterer": "GaussianMixture", "params": {"n_components": 5, "covariance_type": "diag"}},
        {"tag": "mbkm_2", "clusterer": "MiniBatchKMeans", "params": {"n_clusters": 2}},
    ],
    "experiments": [
        "llm_only",
        "cluster_before_llm__gmm_diag_5",
        "cluster_before_llm__mbkm_2",
        "llm_then_cluster__gmm_diag_5",
        "llm_then_cluster__mbkm_2",
    ],
    "note": f"llm_only cache with {BACKEND_TAG} is reused if present; cluster_before_llm uses separate cache files per cluster tag",
}
(ARTIFACT_DIR / "llm_experiment_summary.json").write_text(
    json.dumps(summary_payload, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Saved comparative results to", ARTIFACT_DIR)


In [ ]:
# Extra cell: all ML_baseline.ipynb models with baseline/default parameters, no tuning

from sklearn.base import clone
from sklearn.ensemble import BaggingRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

RUN_BASELINE_PARAM_CV = True  # если будет долго, поставь False

ML_BASELINE_PIPELINES_FULL = [
    (
        "Scaled_Lasso",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("Lasso", Lasso(random_state=seed)),
        ]),
    ),
    (
        "Scaled_Elastic",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("Elastic", ElasticNet(random_state=seed)),
        ]),
    ),
    (
        "Scaled_RF_reg",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("RF", RandomForestRegressor(random_state=seed)),
        ]),
    ),
    (
        "Scaled_ET_reg",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("ET", ExtraTreesRegressor(random_state=seed)),
        ]),
    ),
    (
        "Scaled_BR_reg",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("BR", BaggingRegressor(random_state=seed)),
        ]),
    ),
    (
        "Scaled_DT_reg",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("DT_reg", DecisionTreeRegressor(random_state=seed)),
        ]),
    ),
    (
        "Scaled_Ridge",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("Ridge", Ridge(random_state=seed)),
        ]),
    ),
    (
        "Scaled_SVR",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("SVR", SVR(kernel="linear", C=1e2)),
        ]),
    ),
    (
        "Scaled_Hub-Reg",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("Hub-Reg", HuberRegressor()),
        ]),
    ),
    (
        "Scaled_BayRidge",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("BR", BayesianRidge()),
        ]),
    ),
    (
        "Scaled_KNN_reg",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("KNN_reg", KNeighborsRegressor()),
        ]),
    ),
    (
        "Scaled_Gboost-Reg",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("GBoost-Reg", GradientBoostingRegressor(random_state=seed)),
        ]),
    ),
    (
        "Scaled_XGB_reg",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("XGBR", XGBRegressor(random_state=seed, verbosity=0)),
        ]),
    ),
    (
        "Scaled_RFR_PCA",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("PCA", PCA(n_components=3)),
            ("RF", RandomForestRegressor(random_state=seed)),
        ]),
    ),
    (
        "Scaled_XGBR_PCA",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("PCA", PCA(n_components=3)),
            ("XGB", XGBRegressor(random_state=seed, verbosity=0)),
        ]),
    ),
]

def _baseline_param_cv_mae(estimator, Xtr_aug):
    if not RUN_BASELINE_PARAM_CV:
        return np.nan, np.nan

    fold_maes = []
    cv = TimeSeriesSplit(n_splits=3)

    for tr_idx, va_idx in cv.split(Xtr_aug):
        Xtr_fold = Xtr_aug.iloc[tr_idx]
        ytr_fold = meta_ytr0.iloc[tr_idx]
        Xva_fold = Xtr_aug.iloc[va_idx]
        yva_fold = meta_ytr0.iloc[va_idx]

        est = clone(estimator)
        est.fit(Xtr_fold, ytr_fold)
        pred = est.predict(Xva_fold)
        fold_maes.append(mean_absolute_error(yva_fold, pred))

    return float(np.mean(fold_maes)), float(np.std(fold_maes))

def eval_all_ml_baseline_models_for_variant(Xtr_aug, Xte_typ_aug, Xte_full_aug, experiment):
    rows = []

    for model_name, model in ML_BASELINE_PIPELINES_FULL:
        print(f"[baseline params] {experiment} | {model_name}", flush=True)
        t0 = time.time()

        est = clone(model)
        cv_mae_mean, cv_mae_std = _baseline_param_cv_mae(est, Xtr_aug)

        est.fit(Xtr_aug, meta_ytr0)
        pred_typ = est.predict(Xte_typ_aug)
        pred_full = est.predict(Xte_full_aug)

        rows.append({
            "experiment": experiment,
            "model": model_name,
            "cv_MAE_mean": cv_mae_mean,
            "cv_MAE_std": cv_mae_std,
            "holdout_typical_MAE": float(mean_absolute_error(meta_yte_typ0, pred_typ)),
            "holdout_typical_RMSE": float(np.sqrt(mean_squared_error(meta_yte_typ0, pred_typ))),
            "holdout_typical_MdAE": float(median_absolute_error(meta_yte_typ0, pred_typ)),
            "holdout_full_MAE": float(mean_absolute_error(meta_yte0, pred_full)),
            "holdout_full_RMSE": float(np.sqrt(mean_squared_error(meta_yte0, pred_full))),
            "holdout_full_MdAE": float(median_absolute_error(meta_yte0, pred_full)),
            "fit_seconds": float(time.time() - t0),
            "params": json.dumps(extract_model_params(est), ensure_ascii=False),
        })

    return pd.DataFrame(rows)

baseline_param_frames = []

baseline_param_frames.append(
    eval_all_ml_baseline_models_for_variant(
        meta_Xtr0,
        meta_Xte_typ0,
        meta_Xte0,
        "pure_ml_baseline_params"
    )
)

baseline_param_frames.append(
    eval_all_ml_baseline_models_for_variant(
        Xtr_llm_only,
        Xte_llm_only_typ,
        Xte_llm_only_full,
        "llm_only__baseline_params"
    )
)

for tag, bundle in cluster_before_llm_variants.items():
    baseline_param_frames.append(
        eval_all_ml_baseline_models_for_variant(
            bundle["Xtr"],
            bundle["Xte_typ"],
            bundle["Xte_full"],
            f"cluster_before_llm__{tag}__baseline_params"
        )
    )

for tag, bundle in llm_then_cluster_variants.items():
    baseline_param_frames.append(
        eval_all_ml_baseline_models_for_variant(
            bundle["Xtr"],
            bundle["Xte_typ"],
            bundle["Xte_full"],
            f"llm_then_cluster__{tag}__baseline_params"
        )
    )

all_baseline_param_results = pd.concat(baseline_param_frames, ignore_index=True)

all_baseline_param_results = all_baseline_param_results.sort_values(
    ["holdout_typical_MAE", "holdout_full_MAE", "cv_MAE_mean"],
    na_position="last"
).reset_index(drop=True)

save_path = ARTIFACT_DIR / "all_variants_all_ml_baseline_models_default_params.csv"
all_baseline_param_results.to_csv(save_path, index=False)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

display(all_baseline_param_results)

print("Saved:", save_path)
print("Rows:", len(all_baseline_param_results))


In [ ]:
# Extra cell: all ML_baseline.ipynb models with baseline/default parameters, no tuning

from sklearn.base import clone
from sklearn.ensemble import BaggingRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

RUN_BASELINE_PARAM_CV = True  # если будет долго, поставь False

ML_BASELINE_PIPELINES_FULL = [
    (
        "Scaled_Lasso",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("Lasso", Lasso(random_state=seed)),
        ]),
    ),
    (
        "Scaled_Elastic",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("Elastic", ElasticNet(random_state=seed)),
        ]),
    ),
    (
        "Scaled_RF_reg",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("RF", RandomForestRegressor(random_state=seed)),
        ]),
    ),
    (
        "Scaled_ET_reg",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("ET", ExtraTreesRegressor(random_state=seed)),
        ]),
    ),
    (
        "Scaled_BR_reg",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("BR", BaggingRegressor(random_state=seed)),
        ]),
    ),
    (
        "Scaled_DT_reg",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("DT_reg", DecisionTreeRegressor(random_state=seed)),
        ]),
    ),
    (
        "Scaled_Ridge",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("Ridge", Ridge(random_state=seed)),
        ]),
    ),
    (
        "Scaled_SVR",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("SVR", SVR(kernel="linear", C=1e2)),
        ]),
    ),
    (
        "Scaled_Hub-Reg",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("Hub-Reg", HuberRegressor()),
        ]),
    ),
    (
        "Scaled_BayRidge",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("BR", BayesianRidge()),
        ]),
    ),
    (
        "Scaled_KNN_reg",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("KNN_reg", KNeighborsRegressor()),
        ]),
    ),
    (
        "Scaled_Gboost-Reg",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("GBoost-Reg", GradientBoostingRegressor(random_state=seed)),
        ]),
    ),
    (
        "Scaled_XGB_reg",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("XGBR", XGBRegressor(random_state=seed, verbosity=0)),
        ]),
    ),
    (
        "Scaled_RFR_PCA",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("PCA", PCA(n_components=3)),
            ("RF", RandomForestRegressor(random_state=seed)),
        ]),
    ),
    (
        "Scaled_XGBR_PCA",
        Pipeline([
            ("Scaler", StandardScaler()),
            ("PCA", PCA(n_components=3)),
            ("XGB", XGBRegressor(random_state=seed, verbosity=0)),
        ]),
    ),
]

def _baseline_param_cv_mae(estimator, Xtr_aug):
    if not RUN_BASELINE_PARAM_CV:
        return np.nan, np.nan

    fold_maes = []
    cv = TimeSeriesSplit(n_splits=3)

    for tr_idx, va_idx in cv.split(Xtr_aug):
        Xtr_fold = Xtr_aug.iloc[tr_idx]
        ytr_fold = meta_ytr0.iloc[tr_idx]
        Xva_fold = Xtr_aug.iloc[va_idx]
        yva_fold = meta_ytr0.iloc[va_idx]

        est = clone(estimator)
        est.fit(Xtr_fold, ytr_fold)
        pred = est.predict(Xva_fold)
        fold_maes.append(mean_absolute_error(yva_fold, pred))

    return float(np.mean(fold_maes)), float(np.std(fold_maes))

def eval_all_ml_baseline_models_for_variant(Xtr_aug, Xte_typ_aug, Xte_full_aug, experiment):
    rows = []

    for model_name, model in ML_BASELINE_PIPELINES_FULL:
        print(f"[baseline params] {experiment} | {model_name}", flush=True)
        t0 = time.time()

        est = clone(model)
        cv_mae_mean, cv_mae_std = _baseline_param_cv_mae(est, Xtr_aug)

        est.fit(Xtr_aug, meta_ytr0)
        pred_typ = est.predict(Xte_typ_aug)
        pred_full = est.predict(Xte_full_aug)

        rows.append({
            "experiment": experiment,
            "model": model_name,
            "cv_MAE_mean": cv_mae_mean,
            "cv_MAE_std": cv_mae_std,
            "holdout_typical_MAE": float(mean_absolute_error(meta_yte_typ0, pred_typ)),
            "holdout_typical_RMSE": float(np.sqrt(mean_squared_error(meta_yte_typ0, pred_typ))),
            "holdout_typical_MdAE": float(median_absolute_error(meta_yte_typ0, pred_typ)),
            "holdout_full_MAE": float(mean_absolute_error(meta_yte0, pred_full)),
            "holdout_full_RMSE": float(np.sqrt(mean_squared_error(meta_yte0, pred_full))),
            "holdout_full_MdAE": float(median_absolute_error(meta_yte0, pred_full)),
            "fit_seconds": float(time.time() - t0),
            "params": json.dumps(extract_model_params(est), ensure_ascii=False),
        })

    return pd.DataFrame(rows)

baseline_param_frames = []

baseline_param_frames.append(
    eval_all_ml_baseline_models_for_variant(
        meta_Xtr0,
        meta_Xte_typ0,
        meta_Xte0,
        "pure_ml_baseline_params"
    )
)

baseline_param_frames.append(
    eval_all_ml_baseline_models_for_variant(
        Xtr_llm_only,
        Xte_llm_only_typ,
        Xte_llm_only_full,
        "llm_only__baseline_params"
    )
)

for tag, bundle in cluster_before_llm_variants.items():
    baseline_param_frames.append(
        eval_all_ml_baseline_models_for_variant(
            bundle["Xtr"],
            bundle["Xte_typ"],
            bundle["Xte_full"],
            f"cluster_before_llm__{tag}__baseline_params"
        )
    )

for tag, bundle in llm_then_cluster_variants.items():
    baseline_param_frames.append(
        eval_all_ml_baseline_models_for_variant(
            bundle["Xtr"],
            bundle["Xte_typ"],
            bundle["Xte_full"],
            f"llm_then_cluster__{tag}__baseline_params"
        )
    )

all_baseline_param_results = pd.concat(baseline_param_frames, ignore_index=True)

all_baseline_param_results = all_baseline_param_results.sort_values(
    ["holdout_typical_MAE", "holdout_full_MAE", "cv_MAE_mean"],
    na_position="last"
).reset_index(drop=True)

save_path = ARTIFACT_DIR / "all_variants_all_ml_baseline_models_default_params.csv"
all_baseline_param_results.to_csv(save_path, index=False)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

display(all_baseline_param_results)

print("Saved:", save_path)
print("Rows:", len(all_baseline_param_results))
